In [70]:
# IMPORTS
from pynq import Overlay, allocate, PL, MMIO
from pynq.lib.video import VideoMode
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
from enum import Enum
import time
import os
import glob

projectDir = "/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator"
PL.reset()
overlay = Overlay(projectDir + "/VivadoHW.bit")
overlay.download()

In [71]:
# SETUP AND DEFINES
class VideoSource(Enum):
    HDMI = (0, 0)
    TPG_COLOR_BAR_BOX = (1, 1)
    TPG_GRID_BOX = (2, 1)
    PS_TO_PL = (3, 2)

IN_WIDTH, IN_HEIGHT = 1280, 720
IMAGES_DIR = projectDir + "/images"
VALID_IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')
available_images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(VALID_IMAGE_EXTENSIONS)] if os.path.exists(IMAGES_DIR) else []
ref_images = [os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.lower().endswith(VALID_IMAGE_EXTENSIONS) and '_ref' in f.lower()] if os.path.exists(IMAGES_DIR) else []

tpg = overlay.v_tpg_0
vdma = overlay.dma_block.axi_vdma_0
dma = overlay.dma_block.axi_dma_0
buttons = overlay.axi_gpios.axi_gpio_0
imageResolution = overlay.axi_gpios.axi_gpio_1
threshold_gpio = overlay.axi_gpios.axi_gpio_2
image_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='100%'))
stream_switch = overlay.orb_pipeline.axis_switch_0

current_ps_frame = None
ps_to_pl_selected_image = 0
selected_source = None
current_desc_count = 0
frame_count = 0


In [72]:
threshold_gpio.channel1.write(50, 0xFFFFFFFF)
dummy_desc_buf = allocate(shape=(65536,), dtype=np.uint32) 

current_kpts = []
current_descs = None

SCALE_FACTOR = 2  #TODO scale up in IP when we have multiple 

def parse_descriptors(buffer, num_corners):
    kpts = []
    descs = []
    
    for i in range(num_corners):
        offset = i * 16
        meta = int(buffer[offset + 8])
        
        raw_x = meta & 0xFFFF
        raw_y = (meta >> 16) & 0xFFFF
        
        final_x = raw_x * SCALE_FACTOR
        final_y = raw_y * SCALE_FACTOR
        
        desc_words = buffer[offset : offset + 8]
        desc_bytes = np.array(desc_words, dtype=np.uint32).view(np.uint8)
        
        kpts.append(cv2.KeyPoint(float(final_x), float(final_y), 31))
        descs.append(desc_bytes)
        
    if len(descs) > 0:
        descs = np.array(descs, dtype=np.uint8)
    else:
        descs = None
        
    return kpts, descs

def process_and_rearm_dma():
    global current_desc_count, frame_count, current_kpts, current_descs
    
    status = dma.read(0x34)
    
    if (status & 0x1) or (status & 0x10):
        dma.write(0x30, 0x4)
        while dma.read(0x30) & 0x4: pass
        dma.recvchannel.start()
        dma.recvchannel.transfer(dummy_desc_buf)
        return
        
    if (status & 0x2):
        transferred_bytes = dma.read(0x58)
        nbrOfDescriptors = transferred_bytes // 64 
        
        if nbrOfDescriptors > 0:
            actual_corners = max(0, nbrOfDescriptors - 1)
            current_desc_count = actual_corners
            
            # --- NEW ASSIGNMENT ---
            current_kpts, current_descs = parse_descriptors(dummy_desc_buf, actual_corners)
            print(f"[Frame {frame_count}] Detected Corners: {actual_corners}")
        else:
            current_desc_count = 0
            current_kpts, current_descs = [], None
            
        dma.recvchannel.transfer(dummy_desc_buf)

def setupVideoSource(new_src: VideoSource):
    global selected_source, ps_to_pl_selected_image, current_ps_frame  
    if selected_source == new_src and new_src != VideoSource.PS_TO_PL:
        return

    tpg.write(0x0000, 0x0)
    stream_switch.write(0x040, 0x80000000)
    stream_switch.write(0x000, 0x2)

    if vdma.readchannel.running: vdma.readchannel.stop()

    selected_source = new_src
    imageResolution.channel1.write(IN_HEIGHT, 0xFFFFFFFF)
    imageResolution.channel2.write(IN_WIDTH, 0xFFFFFFFF)
    tpg.write(0x0010, IN_HEIGHT)
    tpg.write(0x0018, IN_WIDTH)

    if selected_source == VideoSource.TPG_COLOR_BAR_BOX:
        tpg.write(0x0020, 9); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
    elif selected_source == VideoSource.TPG_GRID_BOX:
        tpg.write(0x0020, 11); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)

    if not dma.recvchannel.running: dma.recvchannel.start()
    if dma.recvchannel.idle: dma.recvchannel.transfer(dummy_desc_buf)

    if not vdma.writechannel.running:
        vdma.writechannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
        vdma.writechannel.start()

    vdma.readchannel.mode = VideoMode(IN_WIDTH, IN_HEIGHT, 24)
    vdma.readchannel.start()
    vdma.readchannel.irqframecount = 1

    stream_switch.write(0x040, selected_source.value[1])
    stream_switch.write(0x000, 0x2)

    if selected_source in [VideoSource.TPG_COLOR_BAR_BOX, VideoSource.TPG_GRID_BOX]:
        tpg.write(0x0000, 0x81)
        
    elif selected_source == VideoSource.PS_TO_PL:
        if available_images:
            current_image_path = available_images[ps_to_pl_selected_image]
            print(f"Loading image [{ps_to_pl_selected_image + 1}/{len(available_images)}] for PS_TO_PL: {current_image_path}")
            img = cv2.imread(current_image_path)
            img = cv2.resize(img, (IN_WIDTH, IN_HEIGHT))
            current_ps_frame = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ps_to_pl_selected_image = (ps_to_pl_selected_image + 1) % len(available_images)
            
            buf = vdma.writechannel.newframe()
            np.copyto(buf, current_ps_frame)
            vdma.writechannel.writeframe(buf)
        else:
            print("No images available to load.")
            current_ps_frame = None

    print("done")


if not dma.recvchannel.running: dma.recvchannel.start()
dma.recvchannel.transfer(dummy_desc_buf) 

# --- REFERENCE IMAGE BOOTSTRAP ---
print("Capturing Baseline Reference Descriptors...")
ref_image_path = ref_images[0] 
ps_to_pl_selected_image = available_images.index(ref_image_path)
setupVideoSource(VideoSource.PS_TO_PL)
ref_img_bgr = cv2.imread(ref_image_path)
ref_img_bgr = cv2.resize(ref_img_bgr, (IN_WIDTH, IN_HEIGHT))

print("Waiting for hardware pipeline to stabilize and lock features...")
timeout_counter = 0
stable_frames = 0
last_corner_count = -1

# Wait until the pipeline spits out the exact same number of corners 5 frames in a row
while stable_frames < 5 and timeout_counter < 60:
    while (vdma.readchannel._mmio.read(0x34) & 0x1000) == 0:
        process_and_rearm_dma()
        time.sleep(0.001)
        
    _ = vdma.readchannel.readframe() 
    process_and_rearm_dma()
    
    frame_count += 1
    timeout_counter += 1
    
    if current_desc_count > 0:
        # Check if the corner count has settled to a consistent number
        if current_desc_count == last_corner_count:
            stable_frames += 1
        else:
            stable_frames = 1
            
        last_corner_count = current_desc_count
        print(f"  -> Flush Frame {timeout_counter}: {current_desc_count} corners (Stable for {stable_frames}/5)")
    else:
        stable_frames = 0
        last_corner_count = 0

ref_kpts = current_kpts
ref_descs = current_descs

if ref_descs is None or len(ref_kpts) == 0 or stable_frames < 5:
    print(f"\nWARNING: Pipeline timed out or failed to stabilize. Final count: {current_desc_count} features.")
else:
    print(f"\nSUCCESS: Locked {len(ref_kpts)} features from Reference Image after {timeout_counter} frames.")

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
image_widget.layout.width = '100%' 
display(image_widget)


print("Streaming via VDMA interrupts...")
old_button_value = buttons.read(0)

try:
    while True:
        process_and_rearm_dma()

        while (vdma.readchannel._mmio.read(0x34) & 0x1000) == 0:
            process_and_rearm_dma()
            
            r_status = vdma.readchannel._mmio.read(0x34)
            if r_status & 0x1: 
                vdma.readchannel.stop()
                vdma.readchannel.start()
                vdma.readchannel.irqframecount = 1
                
            if r_status & 0x8980:
                vdma.readchannel._mmio.write(0x34, 0x8980)
            time.sleep(0.001)

        frame = vdma.readchannel.readframe()
        process_and_rearm_dma()

        vis = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        
        if current_descs is not None and ref_descs is not None and len(current_kpts) > 0 and len(ref_kpts) > 0:
            matches = bf.match(ref_descs, current_descs)
            
            good_matches = [m for m in matches if m.distance < 60]
            good_matches = sorted(good_matches, key=lambda x: x.distance)
            
            vis = cv2.drawMatches(ref_img_bgr, ref_kpts, vis, current_kpts, good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
            
            cv2.putText(vis, f"Matches: {len(good_matches)}", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        else:
            cv2.putText(vis, f"Descriptors: {current_desc_count}", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
        
        _, jpeg = cv2.imencode('.jpg', vis)
        image_widget.value = jpeg.tobytes()
        
        button_value = buttons.read(0)
        if button_value != old_button_value and button_value > 0:
            if button_value & 0x1: setupVideoSource(VideoSource.HDMI)
            elif button_value & 0x2: setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
            elif button_value & 0x4: setupVideoSource(VideoSource.TPG_GRID_BOX)
            elif button_value & 0x8: setupVideoSource(VideoSource.PS_TO_PL)
        
        old_button_value = button_value
        frame_count += 1
        
except KeyboardInterrupt:
    print("\nStopping stream...")
finally:
    tpg.write(0x00, 0x0)
    if vdma.readchannel.running: vdma.readchannel.stop()
    if vdma.writechannel.running: vdma.writechannel.stop()
    dummy_desc_buf.close()
    print("Stream stopped cleanly.")

Capturing Baseline Reference Descriptors...
Loading image [2/5] for PS_TO_PL: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/images/coke_ref.jpg
done
Waiting for hardware pipeline to stabilize and lock features...
[Frame 0] Detected Corners: 0
[Frame 1] Detected Corners: 0
[Frame 1] Detected Corners: 0
[Frame 1] Detected Corners: 0
[Frame 1] Detected Corners: 1951
  -> Flush Frame 2: 1951 corners (Stable for 1/5)
[Frame 2] Detected Corners: 1951
  -> Flush Frame 3: 1951 corners (Stable for 2/5)
[Frame 3] Detected Corners: 98
  -> Flush Frame 4: 98 corners (Stable for 1/5)
[Frame 4] Detected Corners: 98
  -> Flush Frame 5: 98 corners (Stable for 2/5)
[Frame 5] Detected Corners: 98
  -> Flush Frame 6: 98 corners (Stable for 3/5)
[Frame 6] Detected Corners: 98
  -> Flush Frame 7: 98 corners (Stable for 4/5)
[Frame 7] Detected Corners: 98
  -> Flush Frame 8: 98 corners (Stable for 5/5)

SUCCESS: Locked 98 features from Reference Image after 8 frames.


Image(value=b'', format='jpeg', layout="Layout(width='100%')")

Streaming via VDMA interrupts...
[Frame 8] Detected Corners: 98
[Frame 9] Detected Corners: 98
[Frame 10] Detected Corners: 98
[Frame 11] Detected Corners: 98
[Frame 12] Detected Corners: 98
[Frame 13] Detected Corners: 98
[Frame 14] Detected Corners: 98
[Frame 15] Detected Corners: 98
[Frame 16] Detected Corners: 98
[Frame 17] Detected Corners: 98
[Frame 18] Detected Corners: 98
[Frame 19] Detected Corners: 98
[Frame 20] Detected Corners: 98
[Frame 21] Detected Corners: 98
[Frame 22] Detected Corners: 98
[Frame 23] Detected Corners: 98
Loading image [3/5] for PS_TO_PL: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/images/park.jpg
done
[Frame 24] Detected Corners: 98
[Frame 24] Detected Corners: 98
[Frame 25] Detected Corners: 650
[Frame 26] Detected Corners: 650
[Frame 27] Detected Corners: 650
[Frame 28] Detected Corners: 650
Loading image [4/5] for PS_TO_PL: /home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/images/park_and_coke.jpg
done
[Frame 29] Detected Corners: 650
[

In [14]:
def debug_pipeline():
    print("=== PIPELINE HARDWARE STATUS ===")
    
    # 1. DMA Status (r_brief descriptors)
    print("\n--- DMA (S2MM - Descriptors) ---")
    dma_status = dma.read(0x34)
    print(f"Status Register: {hex(dma_status)}")
    if dma_status & 0x1: print(" [!] HALTED - DMA is stopped.")
    if dma_status & 0x2: print(" [*] IDLE - Waiting for data.")
    if dma_status & 0x10: print(" [FATAL] INTERNAL ERROR (0x5011) - Buffer overflow or TLAST mismatch!")
    if dma_status & 0x20: print(" [FATAL] SLAVE ERROR")
    if dma_status & 0x40: print(" [FATAL] DECODE ERROR")
    if not (dma_status & 0x71): print(" [+] RUNNING - Processing descriptors normally.")

    # 2. VDMA Read Status (Video to PS)
    print("\n--- VDMA Read Channel (S2MM - Video to PS) ---")
    vdma_r_status = vdma.readchannel._mmio.read(0x34)
    print(f"Status Register: {hex(vdma_r_status)}")
    if vdma_r_status & 0x1: print(" [!] HALTED - Read channel is stopped.")
    if vdma_r_status & 0x10: print(" [FATAL] INTERNAL ERROR - AXI stream crashed.")
    if vdma_r_status & 0x80: print(" [!] SOF EARLY ERROR - Frame started before previous finished.")
    if vdma_r_status & 0x100: print(" [!] EOL EARLY ERROR - Line ended too soon (Width mismatch).")
    if vdma_r_status & 0x4000: print(" [!] SOF LATE ERROR - New frame didn't start when expected.")
    if not (vdma_r_status & 0x4191): print(" [+] RUNNING - Capturing video normally.")

    # 3. VDMA Write Status (PS to Video)
    print("\n--- VDMA Write Channel (MM2S - PS to PL) ---")
    vdma_w_status = vdma.writechannel._mmio.read(0x04)
    print(f"Status Register: {hex(vdma_w_status)}")
    if vdma_w_status & 0x1: print(" [!] HALTED")
    if vdma_w_status & 0x10: print(" [FATAL] INTERNAL ERROR")
    if not (vdma_w_status & 0x11): print(" [+] RUNNING - Feeding video normally.")

    # 4. Stream Switch Routing
    print("\n--- AXI Stream Switch ---")
    routing = stream_switch.read(0x040)
    source_map = {0: "HDMI", 1: "TPG", 2: "PS_TO_PL"}
    print(f"Currently routing Si[{routing}] ({source_map.get(routing, 'UNKNOWN')}) to Broadcaster.")

debug_pipeline()

=== PIPELINE HARDWARE STATUS ===

--- DMA (S2MM - Descriptors) ---
Status Register: 0x1000
 [+] RUNNING - Processing descriptors normally.

--- VDMA Read Channel (S2MM - Video to PS) ---
Status Register: 0x10001
 [!] HALTED - Read channel is stopped.

--- VDMA Write Channel (MM2S - PS to PL) ---
Status Register: 0x11001
 [!] HALTED

--- AXI Stream Switch ---
Currently routing Si[0] (HDMI) to Broadcaster.
